# Door state transition detection

**Recipe 85** · Level 2 (Easy) · Decision type: `Choice`

Classify a door as opened, closed, unchanged, or uncertain across chronologically ordered PNG or JPEG frames sampled by Python.

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Build a monitor that describes a door's visible transition across chronological still frames.
One `Choice` returns `opened`, `closed`, `unchanged`, or `uncertain`; Python owns timestamps,
sampling, request limits and simulated observation records. This is an image sequence, not a
native video request. The 80 drawings and all stored probability distributions are authored
synthetic examples, not camera photographs or measured model responses.

A transition is a comparison of sampled states. `Unchanged` means the same visible state at all
sampled times; it cannot rule out an opening and closing between them. An open final frame alone
does not prove that the door opened during the observed interval. Both sampled directions, an
absent door, occlusion or inadequate visibility yield `uncertain`.


## Setup and run mode

Run from this recipe folder. There are 12 validation observations, 12 held-out test observations,
a sampling demo and a reversed-time demo. The header states the mode before any inference.
The same question and image assets work with opt-in live inference using experimental images
and pinned `jev-1.14.0` (S09, S11, S12). The README gives the separate recorder path; exploratory
live output has no reportable held-out interpretation.


In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import confusion_matrix, evaluate_outcomes, select_confidence_threshold
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

apply_style()
helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
session = helpers.Session(backend)
scored = [e for e in examples if e.id in labels]
check = " (a pipeline check, not a Jev result)" if backend.mode == "synthetic" else ""
if backend.mode == "live":
    check = " (unrecorded live exploration; not a reportable result)"
selection = " (a selection step, not a reported result)"
run_header(85, "Door state transition detection", backend=backend, n_examples=len(scored))

## The state

Python receives paths and actual capture times in seconds. It validates every timestamp before
sampling, refusing reversed, duplicate, negative or nonfinite times rather than guessing an
order. It keeps the endpoints and one evenly spaced index, preserves their original times,
and rejects a selected gap above two seconds. This fixed demonstration limit does not ensure
an event cannot be missed.

Jev sees only image references, timestamps and the task. Identifiers, split names, drawing
parameters, filesystem paths and [gold labels](../../docs/glossary.md#gold-label) — the authored
correct judgments used later for scoring — stay outside inference. References identify image
bytes; they carry no class names. The state below belongs to the unscored sampling demo.


In [ ]:
demo = next(e for e in examples if e.id == "d01")
state, questions, assets = helpers.build_request(demo.fields)
print(state)
print("Source timestamps:", [f["timestamp_seconds"] for f in demo.fields["frames"]])
print("Selected timestamps:", [f["timestamp_seconds"] for f in state["frames"]])


def show_frames(fields, sampled=True):
    frames = helpers.sample_frames(fields) if sampled else fields["frames"]
    sheet = Image.new("RGB", (400 * len(frames), 346), "#FEFEFE")
    draw = ImageDraw.Draw(sheet)
    for index, frame in enumerate(frames):
        with Image.open(frame["path"]) as image:
            sheet.paste(image.convert("RGB"), (400 * index, 26))
        draw.text(
            (400 * index + 12, 8), f"{frame['timestamp_seconds']:.2f} seconds", fill="#1E1E1E"
        )
    display(sheet)


show_frames(demo.fields, sampled=False)
show_frames(demo.fields)

## The questions

The single independent judgment fits one `Choice` request. Options and descriptions are fixed
by Python (S02). `Opened` requires shut first and open last; `closed` requires the reverse.
`Unchanged` is a visible state comparison, and `uncertain` is a deferral for insufficient or
contradictory sampled evidence. Camera movement and brightness changes alone are not door events.
Nothing asks the model to calculate elapsed time or select an action.

Choice confidence (S03) rescales the largest probability above the uniform four-option share:
`(p_max - 0.25) / 0.75`. It is not a guarantee of correctness.


In [ ]:
question = questions["transition"]
print(question.instructions)
for option, description in question.criteria.items():
    print(f"{option}: {description}")

## One answer up close

The stored answer is deliberately synthetic. All three selected images show a shut door, so
`unchanged` describes those samples. The five original drawings visibly contain two open states
that sampling omitted. This is temporal aliasing: different intervening events can produce the
same selected observations. No amount of confidence recovers the omitted evidence.


In [ ]:
demo_result = session.decide(demo)
show_answer(demo_result["transition"])
print("Accounting:", run_record(demo_result, mode=backend.mode))

## Python's part

A [confidence gate](../../docs/glossary.md#confidence-gate) sends answers below a chosen cutoff to
[review](../../docs/glossary.md#review), an explicit request for a person to decide. `Uncertain`
always goes there, regardless of confidence. Invalid time sequences stop before any model call.
Accepted observations go to an in-memory `ActionLog`; every entry has `executed=False`. There
is no door actuator. Accepted intervals bound the observation window, not an exact transition time.

The candidate family is the distinct confidence values among determinate validation answers.
Choose the lowest cutoff attaining 100% validation correctness in that eligible subset; this
objective maximizes accepted observations subject to that sample constraint. Freeze it before
opening the test split. Empty eligibility or an unattainable target stops with an error.

The deterministic baseline computes dark-pixel area below grayscale 65 and adjacent absolute
pixel motion within rectangle `(132, 48, 278, 290)`. A growing/shrinking dark gap predicts
`opened`/`closed`; both directions or excessive unexplained motion predict `uncertain`.
Otherwise it predicts `unchanged`. Its four candidates combine gap-change cutoffs 0.12/0.24
and motion cutoffs 12/25. Select maximum raw event correctness on validation, keeping listed
candidate order on ties. This intentionally simple camera-specific rule lacks registration,
illumination correction and semantic occlusion detection.


In [ ]:
validation = [e for e in examples if e.split == "validation"]
val_answers = {e.id: session.decide(e)["transition"] for e in validation}
named = [e for e in validation if val_answers[e.id].choice != "uncertain"]
threshold = select_confidence_threshold(
    [val_answers[e.id].choice == labels[e.id] for e in named],
    [val_answers[e.id].confidence for e in named],
    target_accuracy=1.0,
)
val_outcomes = [helpers.route(e.fields, val_answers[e.id], threshold) for e in validation]
val_metrics = evaluate_outcomes(
    [outcome.accepted for outcome in val_outcomes],
    [val_answers[e.id].choice == labels[e.id] for e in validation],
)
baseline_parameters, baseline_selection = helpers.choose_baseline(validation, labels)
print(
    f"Validation N={len(validation)}, eligible N={len(named)}, frozen threshold={threshold:.4f}{selection}{check}"
)
print(
    f"Validation N={val_metrics.n_total}, answered={val_metrics.n_answered}, coverage={val_metrics.coverage:.4f}, accuracy={val_metrics.accuracy:.4f}{selection}{check}"
)
for parameters, correct_count, n in baseline_selection:
    print(
        f"Validation baseline parameters={parameters}, N={n}, correct={correct_count}, accuracy={correct_count / n:.4f}{selection}{check}"
    )
print(f"Validation frozen baseline parameters={baseline_parameters}{selection}{check}")

In [ ]:
actions, reviews = ActionLog(), ReviewQueue()
demo_outcome = helpers.route(demo.fields, demo_result["transition"], threshold)
helpers.log_outcome(demo.id, demo_outcome, demo_result["transition"], actions, reviews)
reversed_demo = next(e for e in examples if e.id == "d02")
assert session.decide(reversed_demo) is None
reversed_outcome = helpers.route(reversed_demo.fields, None, threshold)
helpers.log_outcome(reversed_demo.id, reversed_outcome, None, actions, reviews)
print("Simulated observations:", actions.to_dicts())
print("Pending reviews:", reviews.to_dicts())

## Evaluation

Now use the frozen choices once on the same 12 test observations for both systems. Raw event
[accuracy](../../docs/glossary.md#accuracy) is correct labels divided by all examples, including
correct `uncertain` predictions. Separately, [coverage](../../docs/glossary.md#coverage) is the
fraction accepted without review; accepted accuracy uses only accepted observations, and
[risk](../../docs/glossary.md#risk) is the fraction of accepted observations that are wrong.
Abstention is the reviewed fraction. `evaluate_outcomes` receives actual routing flags, so an
uncertain answer never inflates coverage. The baseline also defers every `uncertain` result.

These are synthetic pipeline checks. They cannot establish model performance, confidence
calibration, real camera robustness, or the safety of an unattended physical system.


In [ ]:
test = [e for e in examples if e.split == "test"]
test_answers = {e.id: session.decide(e)["transition"] for e in test}
test_outcomes = [helpers.route(e.fields, test_answers[e.id], threshold) for e in test]
predicted = [test_answers[e.id].choice for e in test]
gold = [labels[e.id] for e in test]
baseline_predicted = [helpers.baseline(e.fields, baseline_parameters) for e in test]
correct = [prediction == expected for prediction, expected in zip(predicted, gold, strict=True)]
baseline_correct = [
    prediction == expected for prediction, expected in zip(baseline_predicted, gold, strict=True)
]
model_metrics = evaluate_outcomes([o.accepted for o in test_outcomes], correct)
baseline_metrics = evaluate_outcomes(
    [p != "uncertain" for p in baseline_predicted], baseline_correct
)
for name, metrics, raw_correct in (
    ("Choice policy", model_metrics, correct),
    ("Pixel baseline", baseline_metrics, baseline_correct),
):
    print(
        f"Test {name}: N={metrics.n_total}, raw event accuracy={sum(raw_correct) / len(raw_correct):.4f}{check}"
    )
    print(
        f"Test {name}: N={metrics.n_total}, accepted={metrics.n_answered}, review={metrics.n_total - metrics.n_answered}, coverage={metrics.coverage:.4f}, abstention={1 - metrics.coverage:.4f}, accepted accuracy={metrics.accuracy:.4f}, risk={metrics.risk:.4f}{check}"
    )
for example, outcome in zip(test, test_outcomes, strict=True):
    helpers.log_outcome(example.id, outcome, test_answers[example.id], actions, reviews)
print(
    "Test per-example observations (synthetic labels and replay answers):"
    if backend.mode == "synthetic"
    else "Test per-example observations:"
)
for example, prediction, pixel, outcome in zip(
    test, predicted, baseline_predicted, test_outcomes, strict=True
):
    print(
        example.id,
        "gold=",
        labels[example.id],
        "Choice=",
        prediction,
        "pixel=",
        pixel,
        "accepted=",
        outcome.accepted,
        "reason=",
        outcome.reason,
    )

In [ ]:
matrix = confusion_matrix(gold, predicted, labels=list(helpers.OPTIONS))
print(f"Raw event confusion counts, test N={len(test)}{check}")
print("Row/column labels:", list(helpers.OPTIONS))
print(matrix)
plot_confusion_matrix(matrix, list(helpers.OPTIONS), title=f"Test events, N={len(test)}{check}")

### The accepted shadow error

In the synthetic fixtures, the held-out shadow case preserves a shut panel but adds a dark vertical region. Its authored
answer confidently says `opened`, and the validation-frozen gate accepts it. The visible
failure is intentional: confidence cannot distinguish this wrong answer from a correct one.
A person reviewing the samples can inspect the stable panel outline. Improving the prompt or
baseline after seeing this example would require a fresh held-out set for another result.


In [ ]:
wrong_accepted = [
    e.id
    for e, outcome, ok in zip(test, test_outcomes, correct, strict=True)
    if outcome.accepted and not ok
]
print(f"Accepted wrong observations, test N={len(test)}: {wrong_accepted}{check}")
shadow = next(e for e in test if e.id == "t10")
show_frames(shadow.fields)
show_answer(test_answers[shadow.id])
print("Final logical request count:", session.requests)
print("Pending reviews (demos and test):", len(reviews.pending()))

## What was and was not measured

The cell prints mode, N for each split, and the returned model and capture dates when responses
are recorded. Synthetic responses have no model capture; unknown tokens, inference latency and
cost remain `None`. The recorder retains actual returned model/date/usage. This notebook does
not time inference or infer a bill, and replay time is never relabeled as model latency.

Only sampled states were judged. No exact transition instant, elapsed duration of an opening,
unobserved intermediate event, physical action or real-camera reliability was measured. Rendered
pixels differ across splits, but paired scenarios reuse drawing templates, nuisance families
and layouts. This is not a holdout of unseen scene families.


In [ ]:
if backend.mode == "synthetic":
    print(f"N: {len(validation)} validation, {len(test)} test, 2 unscored demos.")
    print(
        f"Provenance: synthetic. N={len(validation)} validation, N={len(test)} test, 2 unscored demos. Not measured live: a pipeline check, not a Jev result."
    )
elif backend.mode == "recorded":
    print(
        f"Provenance: recorded. Model={backend.model}; dates={list(backend.recorded_dates)}; N={len(validation)} validation selection, N={len(test)} held-out test, 2 unscored demos."
    )
else:
    dates = sorted({r["transition"].provenance.date for r in session.cache.values()})
    print(
        f"Provenance: live. Returned model={backend.model}; dates={dates}; N={len(validation)} validation, N={len(test)} test. Unrecorded exploration is not reportable."
    )
print("Example accounting row:", run_record(demo_result, mode=backend.mode))
print("Measured inference latency=None; measured cost=None.")

## Next steps

Try irregular capture intervals or additional frames, then document which events the sampling
can miss before changing any limits. Build a new validation/test collection from consented
camera footage before making a claim about real doors. Keep decoded frame timestamps; file
order alone is not time. An alignment-aware pixel baseline would address camera motion, but
should be selected and frozen with a fresh test collection.

Continue with [sentiment classification](../01-sentiment-classification/) for typed confidence
and [tool outcome verification](../29-tool-outcome-verification/) for comparing observations
with a claimed action. Local synthetic execution is a build check; genuine Opus review and
main integration checks remain separate requirements.
